In [ ]:
# --- INSTALLATION ROBUSTE (VERSIONS FIGÉES) ---
print("⏳ Installation des cœurs IA...")
# On force les versions qui marchent ensemble pour éviter les conflits
!pip install -q langchain==0.2.11 langchain-community==0.2.10 langchain-core==0.2.23 langchain-text-splitters==0.2.2 langchain-huggingface==0.0.3
!pip install -q faiss-cpu sentence-transformers bitsandbytes accelerate transformers

print("⏳ Installation des outils Vision & Bureautique...")
# Outils pour lire les PDF, Excel, ODS, Images...
!pip install -q pymupdf4llm pandas openpyxl odfpy docx2txt python-pptx reportlab pdf2docx
!pip install -q pytesseract pdf2image pillow

print("🔧 Installation du module LiteLLM...")
# On installe spécifiquement l'extension requise
!pip install -q "smolagents[litellm]"
print("✅ Installation terminée.")

print("⏳ Installation Agent & Web...")
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q -U smolagents
!pip install -q -U duckduckgo-search markdownify requests
!pip install -q ddgs

# --- INSTALLATION SYSTÈME (LINUX) ---
print("⏳ Installation des outils système (LibreOffice, OCR)...")
!sudo apt-get update -q
!sudo apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-fra > /dev/null
!sudo apt-get install -y libreoffice-core libreoffice-writer libreoffice-calc libreoffice-impress --no-install-recommends > /dev/null

print("✅ TERMINÉ. >>> REDÉMARREZ LA SESSION MAINTENANT (Menu Exécution > Redémarrer la session) <<<")

In [ ]:
import os
import io
import warnings
import subprocess
import pandas as pd
import docx2txt
import pymupdf4llm
import google.generativeai as genai
from PIL import Image
from pptx import Presentation
from openpyxl import load_workbook
from docx import Document
from openpyxl.styles import Font, Alignment

# Imports IA corrigés
from smolagents import CodeAgent, LiteLLMModel, Tool, tool, DuckDuckGoSearchTool, VisitWebpageTool
from langchain_community.vectorstores import FAISS
# On utilise Embeddings LOCAL (CPU) pour la stabilité, et Endpoint (API) pour l'intelligence
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

warnings.filterwarnings("ignore")

In [ ]:
# ==========================================
# 🔐 ZONE DES CLÉS (REMETS TES CLÉS ICI)
# ==========================================
MA_CLE_HUGGINGFACE = "mes_clees" 
CLE_GOOGLE_VISION  = "mes_clees" 
# ==========================================

print("🚀 DÉMARRAGE SYSTÈME (VERSION FINALE STABLE)...")

In [ ]:
# --- 1. CONFIGURATION CERVEAU (LITELLM) ---
try:
    # On utilise LiteLLMModel qui est le plus fiable actuellement
    model_agent = LiteLLMModel(
        model_id="huggingface/Qwen/Qwen2.5-Coder-32B-Instruct", 
        api_key=MA_CLE_HUGGINGFACE,
        temperature=0.1
    )
    print("🧠 Cerveau connecté (Qwen 32B via LiteLLM).")
except Exception as e:
    print(f"🛑 ERREUR CERVEAU : {e}")
    model_agent = None

In [ ]:
if CLE_GOOGLE_VISION.startswith("AIza"):
    try:
        genai.configure(api_key=CLE_GOOGLE_VISION)
        model_vision_api = genai.GenerativeModel('gemini-1.5-flash')
        print("👁️ Yeux connectés (Google Gemini).")
    except: model_vision_api = None
else: model_vision_api = None

print("✅ CONFIGURATION OK. Tu peux passer à la suite !")

In [ ]:
def analyser_image_avec_google(chemin_image):
    """Envoie l'image à Google pour OCR et description."""
    if not model_vision_api: return "Vision non configurée."
    try:
        img = Image.open(chemin_image)
        prompt = "Transcris tout le texte et décris l'image (tableaux, graphiques)."
        response = model_vision_api.generate_content([prompt, img])
        return f"\n[VISION]: {response.text}\n"
    except: return "Erreur Vision."

def convertir_tout_document(chemin_fichier):
    """Convertit PDF, Office et Images en texte."""
    print(f"📂 Lecture : {chemin_fichier}")
    ext = os.path.splitext(chemin_fichier)[1].lower()
    texte = ""

    if ext in [".jpg", ".png", ".jpeg"]: return analyser_image_avec_google(chemin_fichier)

    # Conversion LibreOffice pour ODT/ODP
    if ext == ".odt":
        subprocess.run(['libreoffice', '--headless', '--convert-to', 'docx', chemin_fichier, '--outdir', '.'], check=True)
        chemin_fichier = chemin_fichier.replace(".odt", ".docx"); ext = ".docx"
    elif ext == ".odp":
        subprocess.run(['libreoffice', '--headless', '--convert-to', 'pptx', chemin_fichier, '--outdir', '.'], check=True)
        chemin_fichier = chemin_fichier.replace(".odp", ".pptx"); ext = ".pptx"

    try:
        if ext == ".pdf": texte = pymupdf4llm.to_markdown(chemin_fichier)
        elif ext == ".docx": texte = docx2txt.process(chemin_fichier)
        elif ext in [".xlsx", ".xls"]:
            xls = pd.ExcelFile(chemin_fichier)
            for nom in xls.sheet_names: 
                texte += f"\n# FEUILLE '{nom}':\n{pd.read_excel(xls, sheet_name=nom).to_markdown(index=False)}\n"
        elif ext == ".pptx":
            prs = Presentation(chemin_fichier)
            for i, slide in enumerate(prs.slides):
                texte += f"\n# SLIDE {i+1}\n"
                for shape in slide.shapes:
                    if hasattr(shape, "text") and shape.text: texte += shape.text + "\n"
                    # Extraction image slide pour Google Vision
                    if shape.shape_type == 13 and model_vision_api:
                        try:
                            blob = shape.image.blob
                            texte += analyser_image_avec_google(io.BytesIO(blob))
                        except: pass
    except Exception as e: return f"Erreur lecture: {e}"
    return texte

In [ ]:
qa_chain_global = None

def initialiser_rag(liste_fichiers):
    global qa_chain_global
    text_data = ""
    for f in liste_fichiers:
        if os.path.exists(f):
            c = convertir_tout_document(f)
            if c: text_data += c + "\n\n"
    
    if not text_data: print("⚠️ RAG Vide."); return

    try: from langchain_text_splitters import MarkdownTextSplitter
    except ImportError: from langchain.text_splitter import MarkdownTextSplitter

    chunks = MarkdownTextSplitter(chunk_size=1000, chunk_overlap=200).split_text(text_data)
    
    print("⚙️ Vectorisation (CPU Local)...")
    # FIX: On utilise le CPU local (pas d'erreur API KeyError)
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        model_kwargs={'device': 'cpu'}
    )
    vectorstore = FAISS.from_texts(chunks, embeddings)
    
    # Le Cerveau RAG utilise l'API
    llm_rag = HuggingFaceEndpoint(
        repo_id="Qwen/Qwen2.5-7B-Instruct", 
        huggingfacehub_api_token=MA_CLE_HUGGINGFACE, 
        temperature=0.1
    )
    
    qa_chain_global = RetrievalQA.from_chain_type(
        llm=llm_rag, chain_type="stuff", retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
        chain_type_kwargs={"prompt": PromptTemplate(template="CONTEXTE:\n{context}\nQUESTION:\n{question}", input_variables=["context", "question"])}
    )
    print("✅ RAG Initialisé.")

In [ ]:
@tool
def outil_rag(question: str) -> str:
    """
    Interroge les documents pour trouver une information.
    
    Args:
        question: La question précise à poser.
    """
    global qa_chain_global
    if qa_chain_global is None: return "Aucun document."
    try: return qa_chain_global.invoke(question)['result']
    except Exception as e: return f"Erreur RAG: {e}"

@tool
def architecte_excel_universel(nom_fichier: str, actions: str) -> str:
    """
    Crée ou modifie un fichier Excel (.xlsx).
    
    Args:
        nom_fichier: Nom du fichier (ex: bilan.xlsx).
        actions: Liste d'actions séparées par ' || ' (ex: 'SUPPRIMER_LIGNE:1 || STYLE:A1:ROUGE').
    """
    if not os.path.exists(nom_fichier): pd.DataFrame().to_excel(nom_fichier, index=False)
    try:
        wb = load_workbook(nom_fichier); ws = wb.active
        for act in actions.split(' || '):
            p = act.strip().split(':')
            cmd = p[0].upper()
            if cmd == 'SUPPRIMER_LIGNE': ws.delete_rows(int(p[1]))
            elif cmd == 'MODIFIER': ws[p[1]] = ":".join(p[2:])
            elif cmd == 'STYLE':
                cell = ws[p[1]]; params = p[2].split('+')
                if 'GRAS' in params: cell.font = Font(bold=True)
                if 'ROUGE' in params: cell.font = Font(color="FF0000")
        wb.save(nom_fichier); return "Succès Excel."
    except Exception as e: return f"Erreur Excel: {e}"

@tool
def editeur_word_universel(nom_fichier: str, action: str, details: str) -> str:
    """
    Modifie un fichier Word (.docx).
    
    Args:
        nom_fichier: Nom du fichier.
        action: 'ajouter_fin' ou 'remplacer'.
        details: Texte à ajouter.
    """
    if not os.path.exists(nom_fichier): Document().save(nom_fichier)
    try:
        doc = Document(nom_fichier)
        if action == 'ajouter_fin': doc.add_paragraph(details)
        doc.save(nom_fichier); return "Succès Word."
    except: return "Erreur Word"

@tool
def editeur_ppt_universel(nom_fichier: str, action: str, details: str) -> str:
    """
    Modifie un PowerPoint (.pptx).
    
    Args:
        nom_fichier: Nom du fichier.
        action: 'ajouter_slide'.
        details: Titre du slide.
    """
    if not os.path.exists(nom_fichier): Presentation().save(nom_fichier)
    try:
        prs = Presentation(nom_fichier)
        if action == 'ajouter_slide': prs.slides.add_slide(prs.slide_layouts[1]).shapes.title.text = details
        prs.save(nom_fichier); return "Succès PPT."
    except: return "Erreur PPT"

# --- 6. CRÉATION AGENT ---
try: search_tool = DuckDuckGoSearchTool(); visit_tool = VisitWebpageTool()
except: search_tool=None; visit_tool=None

if model_agent:
    agent = CodeAgent(
        model=model_agent,
        tools=[outil_rag, architecte_excel_universel, editeur_word_universel, editeur_ppt_universel, search_tool, visit_tool],
        add_base_tools=True,
        additional_authorized_imports=["pandas", "os", "reportlab", "openpyxl", "docx", "pptx", "datetime", "requests", "markdownify", "google.generativeai", "PIL"],
        verbosity_level=2,
        max_steps=12
    )
    print("\n🤖 AGENT DÉFINITIF PRÊT !")
else:
    print("🛑 ERREUR : Vérifie tes clés API !")

In [ ]:
# --- TEST FINAL RAPIDE ---
from reportlab.pdfgen import canvas
pdf_name = "rapport_financier_fictif.pdf"
c = canvas.Canvas(pdf_name); c.drawString(100, 750, "TEST: CA=1M, PDG=Elon Musk"); c.save()
initialiser_rag([pdf_name])

In [ ]:
# Mission Audit
# 1. On charge le fichier généré
fichiers_mission = ['Rapport_Financier_TechNova_2025.pdf']
initialiser_rag(fichiers_mission)

# 2. La Mission
mission_audit = """
Agis comme un Auditeur Financier.
Analyse le document PDF 'Rapport_Financier_TechNova_2025.pdf'.

TACHE 1 : Extraction
Trouve :
- Le Chiffre d'Affaires.
- Le Résultat Net.
- Le nom du PDG.
- Le Ratio de Solvabilité (caché dans le texte).

TACHE 2 : Création Excel
Crée un fichier 'Audit_TechNova.xlsx'.
1. Crée les colonnes 'Indicateur' et 'Valeur'.
2. Remplis avec les données trouvées.
3. Mets la ligne d'en-tête (Ligne 1) en GRAS, ROUGE et CENTRÉ.
"""

print(f"🎯 MISSION AUDIT : {mission_audit}")
try:
    res = agent.run(mission_audit)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# Mission Update Excel (Suppose que le fichier Audit_TechNova.xlsx existe)
mission_update = """
Agis comme un assistant administratif expert.
Je veux que tu mettes à jour le fichier 'Audit_TechNova.xlsx' (ou celui créé précédemment).

TACHE 1 : MISE À JOUR
Double tous les chiffres de la colonne Valeur (Multiplie par 2).
Attention : Ne touche pas aux titres.

TACHE 2 : NOUVEAUX TOTAUX
À la fin du tableau, ajoute une ligne "NOUVEAU TOTAL".
Insère une formule Excel (=SUM(...)) pour faire la somme automatique.
Mets cette ligne en GRAS.
"""

print(f"🎯 MISSION UPDATE : {mission_update}")
try:
    res = agent.run(mission_update)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# Mission LibreOffice ODT/ODP
initialiser_rag([])

mission_libreoffice = """
Agis comme un Secrétaire sous Linux.

1. Crée un nouveau document texte au format LibreOffice (.odt) nommé 'Projet_Libre.odt'.
2. Écris dedans : "Ce fichier a été généré et modifié en format OpenDocument."
3. Applique le style GRAS et TAILLE:24 sur cette phrase.

4. Crée une présentation LibreOffice (.odp) nommée 'Diapo_Libre.odp'.
5. Ajoute une slide avec le titre "Vive l'Open Source".

IMPORTANT : Je veux récupérer des fichiers .odt et .odp à la fin.
"""

print(f"🎯 MISSION LIBREOFFICE : {mission_libreoffice}")
try:
    res = agent.run(mission_libreoffice)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# Mission Vision
# ⚠️ REMPLACE PAR LE NOM DE TA PHOTO ⚠️
nom_fichier_image = 'recette_ancienne.jpg' 

initialiser_rag([nom_fichier_image])

mission_vision = f"""
Agis comme un Archiviste.
Regarde le document image '{nom_fichier_image}'.

TACHE :
"Qu'est-ce qui est marqué dans ce document ?"

Tu dois :
1. Identifier la nature du document.
2. Transcrire ce que tu arrives à lire (ingrédients, instructions...).
Base-toi sur le texte identifié comme [TEXTE MANUSCRIT].
"""

print(f"🎯 MISSION VISION : {mission_vision}")
try:
    res = agent.run(mission_vision)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# Mission 12
# 1. On charge l'image pour que le module Vision l'analyse et la mette en mémoire
# ⚠️ REMPLACE PAR LE NOM EXACT DE TON FICHIER IMAGE ⚠️
nom_image = 'nom_de_ton_image.jpg' 
initialiser_rag([nom_image])

# 2. La Mission Multimodale
mission_12 = f"""
Agis comme un Expert en Présentation Visuelle.

TACHE 1 : ANALYSE
Interroge ta mémoire (RAG) pour savoir ce qui est décrit dans le document image '{nom_image}'.
Récupère une description claire de la scène.

TACHE 2 : CRÉATION POWERPOINT
Crée un fichier PowerPoint nommé 'analayse_image.pptx'.
1. Ajoute une nouvelle diapositive.
2. En TITRE de la diapositive, mets : "Analyse IA".
3. En CONTENU (Texte) de la diapositive, colle la description que tu as trouvée à l'étape 1.

TACHE 3 : INSERTION IMAGE
Sur cette MÊME diapositive (Slide 1), insère l'image originale '{nom_image}'.

Génère le fichier final.
"""

print(f"🎯 MISSION 12 : {mission_12}")
try:
    res = agent.run(mission_12)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# Mission 13
fichiers_mission = ['analayse_image.pptx'] # Le fichier créé à la mission 12
initialiser_rag([]) # Pas besoin de lire le contenu, on va l'écraser

mission_13 = """
Agis comme un Directeur Artistique un peu autoritaire.
Tu dois modifier le fichier 'analayse_image.pptx' (ou .odp).

TACHE :
Sur la première diapositive, je ne veux plus voir la description précédente.
Remplace TOUT le texte existant par cette unique phrase choc :
"ça méritait un rouge"

CONSIGNES DE STYLE :
1. Le texte doit être écrit en MAJUSCULES (Lettres Capitales).
2. Le texte doit être en GRAS.

Utilise ton outil PowerPoint spécialisé pour écraser le texte et appliquer ce style.
"""

print(f"🎯 MISSION 13 : {mission_13}")
try:
    res = agent.run(mission_13)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# PRÉPARATION : GÉNÉRATION DU PDF FICTIF
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4

def creer_pdf_fictif():
    nom = "Rapport_Financier_TechNova_2025.pdf"
    c = canvas.Canvas(nom, pagesize=A4)
    # Page 1
    c.setFont("Helvetica-Bold", 20); c.drawString(100, 750, "RAPPORT ANNUEL 2025 - TECHNOVA")
    c.setFont("Helvetica", 12)
    c.drawString(50, 700, "1. Synthèse des Résultats")
    # Tableau simulé
    y = 650
    data = [("Indicateur", "Valeur"), ("Chiffre d'Affaires", "45 000 000 E"), ("Resultat Net", "8 200 000 E")]
    for k,v in data: c.drawString(50, y, k); c.drawString(300, y, v); y-=20
    # Texte caché
    c.drawString(50, 500, "Le PDG, Monsieur Elon Bezos, est ravi.")
    c.drawString(50, 480, "Ratio de Solvabilité : 22.5%.")
    c.save()
    print(f"✅ Fichier '{nom}' généré.")

creer_pdf_fictif()

In [ ]:
# MISSION 1 & 2 : EXTRACTION PDF -> CRÉATION EXCEL
fichiers_mission = ['Rapport_Financier_TechNova_2025.pdf']
initialiser_rag(fichiers_mission)

mission_1_2 = """
Agis comme un Auditeur Financier.
Analyse le document PDF 'Rapport_Financier_TechNova_2025.pdf'.

TACHE 1 : Extraction
Trouve :
- Le Chiffre d'Affaires.
- Le Résultat Net.
- Le nom du PDG.
- Le Ratio de Solvabilité (caché dans le texte).

TACHE 2 : Création Excel
Crée un fichier 'Audit_TechNova.xlsx'.
1. Crée les colonnes 'Indicateur' et 'Valeur'.
2. Remplis avec les données trouvées.
3. Mets la ligne d'en-tête (Ligne 1) en GRAS, ROUGE et CENTRÉ.
"""

print(f"🎯 MISSION 1 & 2 : {mission_1_2}")
try:
    res = agent.run(mission_1_2)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# MISSION 3 : RECHERCHE FINANCIÈRE COMPLEXE
fichiers_mission = ['Societe-Generale-Pilier-3_T2-2022_FR.pdf']
initialiser_rag(fichiers_mission)

mission_3 = """
Agis comme un Analyste Financier Spécialisé en Banque.
Analyse le document PDF 'Societe-Generale-Pilier-3_T2-2022_FR.pdf'.

TACHE :
Tu dois trouver le montant des fonds propres à la fin de la période (généralement le 30 juin 2022).
Cherche le tableau "KM1" (Key Metrics) et donne-moi les valeurs pour :
1. Les Fonds propres de base de catégorie 1 (CET1).
2. Les Fonds propres totaux.

Cite la page où se trouve l'information.
"""

print(f"🎯 MISSION 3 : {mission_3}")
try:
    res = agent.run(mission_3)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# MISSION 4 : LECTURE EXCEL -> RÉDACTION RAPPORT PDF
# (Utilise le fichier créé à la mission 1 & 2)
initialiser_rag([]) # On vide le RAG, l'agent doit lire l'Excel avec Pandas

mission_4 = """
Agis comme un Directeur Financier.

Tu as le fichier 'Audit_TechNova.xlsx' sur le disque.
Je veux que tu relises ce fichier pour me produire un rapport de synthèse.

1. Analyse les chiffres présents dans cet Excel (CA, Résultat).
2. Calcule la rentabilité (Résultat Net / CA) et donne-moi ton avis.
3. Rédige un document PDF nommé 'Rapport_Final.pdf' qui contient :
   - Un titre propre.
   - Ton analyse des chiffres.
   - Une conclusion sur la santé financière.

Débrouille-toi pour lire le fichier et générer le rapport.
"""

print(f"🎯 MISSION 4 : {mission_4}")
try:
    res = agent.run(mission_4)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# MISSION 5 : FILTRAGE DONNÉES EXCEL/ODS
nom_fichier_sout = 'TER 2023-2024 - Soutenances.xlsx'
initialiser_rag([nom_fichier_sout])

mission_5 = f"""
Agis comme un Secrétaire Universitaire.
Analyse le document '{nom_fichier_sout}'.

TACHE :
L'utilisateur demande : "Cite moi les projets dont LeCapitaine est responsable".

Tu dois :
1. Chercher dans les colonnes "Resp" (Responsable).
2. Trouver toutes les occurrences de "LeCapitaine" ou "H. Le Capitaine".
3. Identifier le "Subject" (Sujet) associé.
4. Me lister uniquement les titres de ces projets.
"""

print(f"🎯 MISSION 5 : {mission_5}")
try:
    res = agent.run(mission_5)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# MISSION 7 : SYNTHÈSE DE DOCUMENT
# (On utilise le même fichier que la mission 5 pour changer)
nom_fichier_a_analyser = 'TER 2023-2024 - Soutenances.xlsx'
initialiser_rag([nom_fichier_a_analyser])

mission_7 = f"""
Agis comme un Assistant Personnel.
Je te donne le fichier '{nom_fichier_a_analyser}'.

TACHE :
Réponds simplement à cette question : "Qu'est-ce qui est marqué dans ce document ?"

1. Identifie le type de document (Planning ? Facture ?).
2. Donne les infos clés (Dates, Lieux).
3. Cite 2 ou 3 exemples de sujets pour illustrer.
"""

print(f"🎯 MISSION 7 : {mission_7}")
try:
    res = agent.run(mission_7)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# MISSION 9 : RECHERCHE WEB & CRÉATION WORD
initialiser_rag([])

mission_9 = """
Agis comme un Expert Linux.

TACHE 1 : RECHERCHE WEB
Cherche la procédure pour "Installer Ubuntu en Dual Boot sur Windows".
Trouve les grandes étapes clés.

TACHE 2 : CRÉATION DU GUIDE
Crée un fichier Word nommé 'ubuntu.docx'.
1. Titre : "GUIDE INSTALLATION UBUNTU" (Style : GRAS + CENTRÉ + TAILLE:24).
2. Intro : Avertissement sur la sauvegarde des données (GRAS + ROUGE).
3. Les Étapes : Rédige les étapes trouvées.
"""

print(f"🎯 MISSION 9 : {mission_9}")
try:
    res = agent.run(mission_9)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# MISSION 10 : MISE EN FORME WORD
# (Nécessite que la Mission 9 soit faite)
initialiser_rag([])

mission_10 = """
Agis comme un assistant bureautique.
Reprends le fichier 'ubuntu.docx'.

Je veux améliorer la mise en forme.
Trouve toutes les fois où sont écrits les mots "Windows", "Linux" et "Ubuntu".
Mets-les systématiquement en GRAS et en ITALIQUE pour qu'ils ressortent bien.

Fais ça proprement.
"""

print(f"🎯 MISSION 10 : {mission_10}")
try:
    res = agent.run(mission_10)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# MISSION 11 : ANALYSE PRÉSENTATION TECHNIQUE (ODP)
nom_ppt = 'Présentation générale Capstone SPARQL LLM.odp'
initialiser_rag([nom_ppt])

mission_11 = f"""
Agis comme un Consultant Technique.
Analyse la présentation '{nom_ppt}'.

TACHE 1 : RECHERCHE
Trouve et explique le rôle de "SCHEMAORG" (écrit exactement ainsi en majuscules).

TACHE 2 : SYNTHÈSE & RAPPORT
Crée un PDF 'résume_powerpoint.pdf' contenant :
1. Titre "ANALYSE PROJET SPARQL" (Gras, Centré).
2. Un paragraphe sur le rôle de SCHEMAORG.
3. Un résumé global du projet.
"""

print(f"🎯 MISSION 11 : {mission_11}")
try:
    res = agent.run(mission_11)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# MISSION LIBREOFFICE : CRÉATION DOCUMENTS OPEN SOURCE
initialiser_rag([])

mission_libre = """
Agis comme un Secrétaire Linux.

1. Crée un document texte LibreOffice (.odt) nommé 'Projet_Libre.odt'.
   - Écris : "Document généré en OpenDocument." (GRAS, TAILLE:24).

2. Crée une présentation LibreOffice (.odp) nommée 'Diapo_Libre.odp'.
   - Ajoute une slide avec le titre "Vive l'Open Source".

Je veux uniquement des fichiers .odt et .odp à la fin.
"""

print(f"🎯 MISSION LIBREOFFICE : {mission_libre}")
try:
    res = agent.run(mission_libre)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# MISSION 12 : IMAGE VERS POWERPOINT
nom_image = 'nom_de_ton_image.jpg' # ⚠️ METS LE VRAI NOM DE TON IMAGE ICI
initialiser_rag([nom_image])

mission_12 = f"""
Agis comme un Expert Visuel.

TACHE 1 : ANALYSE
Récupère la description de l'image '{nom_image}' via ta mémoire.

TACHE 2 : CRÉATION POWERPOINT
Crée 'analayse_image.pptx'.
1. Ajoute une slide.
2. Titre : "Analyse IA".
3. Texte : Colle la description de l'image.
4. Insère l'image '{nom_image}' sur cette même slide.
"""

print(f"🎯 MISSION 12 : {mission_12}")
try:
    res = agent.run(mission_12)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")

In [ ]:
# MISSION 13 : MODIFICATION PPT AVEC STYLE
# (Utilise le fichier créé à la mission 12)
initialiser_rag([])

mission_13 = """
Agis comme un Directeur Artistique.
Modifie le fichier 'analayse_image.pptx'.

TACHE :
Sur la première diapositive, remplace TOUT le texte existant par :
"ça méritait un rouge"

CONSIGNES DE STYLE :
1. Le texte doit être en MAJUSCULES.
2. Le texte doit être en GRAS.
Utilise ton outil pour écraser le texte.
"""

print(f"🎯 MISSION 13 : {mission_13}")
try:
    res = agent.run(mission_13)
    print(f"\n💡 RÉSULTAT :\n{res}")
except Exception as e:
    print(f"🛑 Erreur : {e}")